# CardioTrace 04 — SHAP Interpretability

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv; load_dotenv('../.env')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 40)
from src.etl import get_engine
engine = get_engine()
df = pd.read_sql('SELECT * FROM mart.mart_cv_master', engine)
print(f'mart_cv_master: {df.shape[0]:,} rows x {df.shape[1]} cols')

SHAP (TreeExplainer) attributes each prediction to its drivers — the global summary shows which risk factors matter most across the population.

In [ ]:
import shap, src.model as M, src.analysis as A
from run_pipeline import NUM_FEATURES, CAT_FEATURES
feats = [f for f in NUM_FEATURES+CAT_FEATURES if f in df.columns]
cat = [f for f in CAT_FEATURES if f in df.columns]
res = M.train_target(df, 'has_any_cvd', feats, cat)
pipe = res['xgboost']['pipeline']
X = df.loc[df['has_any_cvd'].notna(), feats].sample(2000, random_state=42)
Xt = pipe.named_steps['prep'].transform(X)
names = [f.split('__')[-1] for f in pipe.named_steps['prep'].get_feature_names_out()]
sv = shap.TreeExplainer(pipe.named_steps['model']).shap_values(Xt)
shap.summary_plot(sv, Xt, feature_names=names, max_display=12)